# uxplain demo — Classification

End-to-end walkthrough of the classification pipeline on synthetic data:

1. Generate a 4-class problem with `sklearn.datasets.make_classification`.
2. Fit `UncertaintyExplanationPipeline` and inspect prediction **sets** (instead of intervals).
3. Explain `set_size` (default metric for classification) with SHAP.
4. Swap the **uncertainty metric** to `credibility` and `confidence`.
5. Try the **class-conditional** conformal method for per-class coverage guarantees.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import uxplain
from uxplain import UncertaintyExplanationPipeline

print(f"uxplain {uxplain.__version__}")

RNG = 42

## 1. Simulated data

Four classes, 10 features (6 informative), 1 500 samples. Stratified split so each class is represented in train and test.

In [ ]:
X_raw, y = make_classification(
    n_samples=1500,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=4,
    n_clusters_per_class=2,
    class_sep=1.2,
    random_state=RNG,
)
feature_names = [f"x{i}" for i in range(X_raw.shape[1])]
X = pd.DataFrame(X_raw, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RNG, stratify=y
)
print(f"train: {X_train.shape}  test: {X_test.shape}")
print(f"classes: {np.unique(y)} (counts: {np.bincount(y)})")

## 2. Quickstart

`task="classification"` is auto-detected from the model. Defaults: `confidence=0.9`, `conformal_method="standard"`, `uncertainty_metric="set_size"`, `xai_method="shap"`.

In [ ]:
pipeline = UncertaintyExplanationPipeline(
    model=RandomForestClassifier(n_estimators=200, random_state=RNG),
    task="classification",
    confidence=0.9,
    random_state=RNG,
)
pipeline.fit(X_train, y_train)

### Prediction sets and set-size distribution

For each sample, the model emits a *set* of plausible classes. A set of size 1 = confident; larger sets = the model is uncertain among multiple classes.

In [ ]:
result = pipeline.explain(X_test.iloc[:300], show_plots=False)

# Coverage: is the true class inside the predicted set?
in_set = result.prediction_set[np.arange(len(result.prediction_set)), y_test[:300]]
print(f"Empirical coverage: {in_set.mean():.2%}   (target ≥ 90 %)")

print("\nSet-size distribution:")
sizes, counts = np.unique(result.set_size, return_counts=True)
for s, c in zip(sizes, counts):
    pct = c / len(result.set_size)
    print(f"  size {s}: {c:3d}  ({pct:.1%})")

### Per-sample inspection

For a handful of test samples, look at the true class, the predicted set, and the conformal p-values per class.

In [ ]:
rows = []
for i in range(8):
    pred_classes = result.classes[result.prediction_set[i]].tolist()
    rows.append({
        "sample": i,
        "true_class": int(y_test[i]),
        "predicted_set": pred_classes,
        "set_size": int(result.set_size[i]),
        "p_values": result.p_values[i].round(3).tolist(),
    })
pd.DataFrame(rows)

## 3. Explain `set_size` with SHAP

SHAP attributes each feature's contribution to the **set size** — features that push set size up are increasing the model's uncertainty for that sample.

In [ ]:
_ = pipeline.explain(
    X_test.iloc[:200],
    plot_kind=["beeswarm", "bar", "waterfall"],
    waterfall_index=0,
)

## 4. Different uncertainty metrics

Beyond `set_size`, classification supports:

- `"credibility"` — max p-value across classes. How compatible the top class is with the calibration set.
- `"confidence"` — `1 - second-highest p-value`. How decisively the runner-up class is rejected.

Explaining a different metric changes what the SHAP/PDP/LIME values mean.

In [ ]:
cred_pipeline = UncertaintyExplanationPipeline(
    model=RandomForestClassifier(n_estimators=200, random_state=RNG),
    task="classification",
    uncertainty_metric="credibility",
    random_state=RNG,
)
cred_pipeline.fit(X_train, y_train)
_ = cred_pipeline.explain(X_test.iloc[:200], plot_kind="bar")

## 5. Class-conditional conformal method

`conformal_method="class_cond"` calibrates per class, giving (approximate) per-class coverage instead of marginal coverage. Useful when class importance is asymmetric or class frequencies are imbalanced.

In [ ]:
cc_pipeline = UncertaintyExplanationPipeline(
    model=RandomForestClassifier(n_estimators=200, random_state=RNG),
    task="classification",
    conformal_method="class_cond",
    confidence=0.9,
    random_state=RNG,
)
cc_pipeline.fit(X_train, y_train)
cc_result = cc_pipeline.explain(X_test.iloc[:300], show_plots=False)

in_set_cc = cc_result.prediction_set[np.arange(len(cc_result.prediction_set)), y_test[:300]]
print(f"Marginal coverage (class_cond): {in_set_cc.mean():.2%}\n")

# Per-class coverage
print("Per-class coverage:")
for c in np.unique(y_test[:300]):
    mask = y_test[:300] == c
    cov_c = cc_result.prediction_set[mask, c].mean()
    print(f"  class {c}: {cov_c:.2%}  ({mask.sum()} samples)")

## Recap

- Classification swaps intervals for prediction sets — `result.prediction_set`, `result.p_values`, `result.set_size`, `result.classes`.
- The XAI methods explain a chosen scalar metric (`set_size`, `credibility`, or `confidence`).
- `conformal_method="class_cond"` is the right choice when you need per-class coverage rather than the marginal target.